In [26]:
# Cell 1: 필수 라이브러리 설치
%pip install imagehash pillow requests --quiet


Note: you may need to restart the kernel to use updated packages.


In [27]:
# Cell 2: 라이브러리 import 및 설정
import json
import imagehash
from PIL import Image
import requests
from io import BytesIO
from difflib import SequenceMatcher
import time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import re

# 설정
SIMILARITY_THRESHOLD = 0.5  # 유사도 임계값 (0.5 이상인 것만 Vision API로 비교)
IMAGE_WEIGHT = 0.4  # 이미지 유사도 가중치
PRICE_WEIGHT = 0.2  # 가격 유사도 가중치
TITLE_WEIGHT = 0.4  # 상품명 유사도 가중치
MAX_WORKERS = 10  # 병렬 처리 스레드 수
TOTAL_WEIGHT = IMAGE_WEIGHT + PRICE_WEIGHT + TITLE_WEIGHT

print("✅ 라이브러리 import 완료")
print(f"📊 유사도 임계값: {SIMILARITY_THRESHOLD}")
print(f"⚖️ 가중치 - 이미지: {IMAGE_WEIGHT}, 가격: {PRICE_WEIGHT}, 상품명: {TITLE_WEIGHT}")
print(f"⚡ 병렬 처리: 최대 {MAX_WORKERS}개 스레드\n")


✅ 라이브러리 import 완료
📊 유사도 임계값: 0.5
⚖️ 가중치 - 이미지: 0.4, 가격: 0.2, 상품명: 0.4
⚡ 병렬 처리: 최대 10개 스레드



In [ ]:
# Cell 3: 이미지 해싱 함수 (캐싱 포함)

# 전역 캐시 딕셔너리
_image_hash_cache = {}

def get_image_hash(image_url, timeout=3):
    """
    이미지 URL에서 perceptual hash를 계산합니다 (캐싱 포함).
    
    Args:
        image_url: 이미지 URL
        timeout: 요청 타임아웃 (초)
    
    Returns:
        imagehash.ImageHash 또는 None (실패 시)
    """
    # 캐시 확인
    if image_url in _image_hash_cache:
        return _image_hash_cache[image_url]
    
    try:
        response = requests.get(image_url, timeout=timeout)
        response.raise_for_status()
        img = Image.open(BytesIO(response.content))
        hash_value = imagehash.phash(img)  # perceptual hash 사용
        
        # 캐시에 저장
        _image_hash_cache[image_url] = hash_value
        return hash_value
    except Exception:
        # 실패한 경우 캐시에 저장하지 않아 일시적 오류 시 재시도 가능
        return None


def calculate_image_similarity(url1, url2):
    """
    두 이미지의 유사도를 계산합니다 (0-1, 1에 가까울수록 유사).
    
    Args:
        url1: 첫 번째 이미지 URL
        url2: 두 번째 이미지 URL
    
    Returns:
        float: 유사도 (0.0-1.0)
    """
    hash1 = get_image_hash(url1)
    hash2 = get_image_hash(url2)
    
    if hash1 is None or hash2 is None:
        return 0.0
    
    # Hamming distance 계산 (0-64)
    difference = hash1 - hash2
    
    # 유사도로 변환 (0-1)
    similarity = 1 - (difference / 64.0)
    return max(0.0, min(1.0, similarity))  # 0-1 범위로 제한


print("✅ 이미지 해싱 함수 정의 완료")
print("   - get_image_hash(): 이미지 URL에서 해시 계산")
print("   - calculate_image_similarity(): 두 이미지의 유사도 계산\n")


✅ 이미지 해싱 함수 정의 완료
   - get_image_hash(): 이미지 URL에서 해시 계산
   - calculate_image_similarity(): 두 이미지의 유사도 계산



In [29]:
# Cell 4: 상품명 유사도 계산 함수

def calculate_title_similarity(title1, title2):
    """
    두 상품명의 유사도를 계산합니다 (0-1, 1에 가까울수록 유사).
    
    Args:
        title1: 첫 번째 상품명
        title2: 두 번째 상품명
    
    Returns:
        float: 유사도 (0.0-1.0)
    """
    if not title1 or not title2:
        return 0.0
    
    # 소문자로 변환하여 비교
    title1_lower = title1.lower().strip()
    title2_lower = title2.lower().strip()
    
    # SequenceMatcher를 사용한 유사도 계산
    similarity = SequenceMatcher(None, title1_lower, title2_lower).ratio()
    
    return similarity


print("✅ 상품명 유사도 계산 함수 정의 완료")
print("   - calculate_title_similarity(): 두 상품명의 유사도 계산\n")


✅ 상품명 유사도 계산 함수 정의 완료
   - calculate_title_similarity(): 두 상품명의 유사도 계산



In [30]:
# Cell 5: 가격 유사도 계산 함수

def parse_price(price_value):
    """문자열/숫자 형태의 가격을 float로 변환합니다."""
    if price_value is None:
        return None
    if isinstance(price_value, (int, float)):
        return float(price_value)
    cleaned = re.sub(r"[^0-9.]", "", str(price_value))
    if not cleaned:
        return None
    try:
        return float(cleaned)
    except ValueError:
        return None


def calculate_price_similarity(price1, price2):
    """두 가격의 유사도(0-1)를 계산합니다."""
    p1 = parse_price(price1)
    p2 = parse_price(price2)

    if p1 is None or p2 is None or max(p1, p2) == 0:
        return 0.0

    diff_ratio = abs(p1 - p2) / max(p1, p2)
    return max(0.0, 1 - diff_ratio)


print("✅ 가격 유사도 계산 함수 정의 완료")
print("   - parse_price(): 가격 문자열을 float로 변환")
print("   - calculate_price_similarity(): 두 가격의 유사도 계산\n")


✅ 가격 유사도 계산 함수 정의 완료
   - parse_price(): 가격 문자열을 float로 변환
   - calculate_price_similarity(): 두 가격의 유사도 계산



In [ ]:
# Cell 5: 종합 유사도 계산 및 필터링 함수

def calculate_combined_similarity(product1, product2):
    """
    이미지, 가격, 상품명 유사도를 종합하여 계산합니다.
    
    Args:
        product1: 첫 번째 상품 정보 (dict)
        product2: 두 번째 상품 정보 (dict)
    
    Returns:
        dict: {
            "combined_similarity": 종합 유사도 (0-1),
            "image_similarity": 이미지 유사도 (0-1),
            "price_similarity": 가격 유사도 (0-1),
            "title_similarity": 상품명 유사도 (0-1)
        }
    """
    # 이미지 유사도 계산
    img_url1 = product1.get("thumbnail_url", "")
    img_url2 = product2.get("thumbnail_url", "")
    image_sim = calculate_image_similarity(img_url1, img_url2) if img_url1 and img_url2 else 0.0
    
    # 상품명 유사도 계산
    title1 = product1.get("title", "")
    title2 = product2.get("title", "")
    title_sim = calculate_title_similarity(title1, title2)
    
    # 가격 유사도 계산 (표시가 우선, 없으면 원가)
    price1 = product1.get("displayed_price") or product1.get("price") or product1.get("original_price")
    price2 = product2.get("displayed_price") or product2.get("price") or product2.get("original_price")
    price_sim = calculate_price_similarity(price1, price2)
    
    # 가중 평균으로 종합 유사도 계산
    weighted_sum = (image_sim * IMAGE_WEIGHT) + (price_sim * PRICE_WEIGHT) + (title_sim * TITLE_WEIGHT)
    combined = weighted_sum / TOTAL_WEIGHT if TOTAL_WEIGHT else 0.0
    
    return {
        "combined_similarity": combined,
        "image_similarity": image_sim,
        "price_similarity": price_sim,
        "title_similarity": title_sim
    }


def filter_similar_products(products_coupang, products_ssadagu, threshold=SIMILARITY_THRESHOLD, max_workers=MAX_WORKERS):
    """
    쿠팡과 싸다구 상품 중 유사한 상품 쌍을 필터링합니다 (병렬 처리).
    
    Args:
        products_coupang: 쿠팡 상품 리스트
        products_ssadagu: 싸다구 상품 리스트
        threshold: 유사도 임계값 (기본값: SIMILARITY_THRESHOLD)
        max_workers: 병렬 처리 스레드 수 (기본값: MAX_WORKERS)
    
    Returns:
        list: 유사한 상품 쌍 리스트
    """
    candidates = []
    
    # 비교 작업 리스트 생성 (썸네일이 있는 것만)
    comparison_tasks = []
    for p1 in products_coupang:
        for p2 in products_ssadagu:
            if p1.get("thumbnail_url") and p2.get("thumbnail_url"):
                comparison_tasks.append((p1, p2))
    
    total_comparisons = len(comparison_tasks)
    total_possible = len(products_coupang) * len(products_ssadagu)
    
    print(f"🔍 총 {len(products_coupang)}개(쿠팡) × {len(products_ssadagu)}개(싸다구) = {total_possible}개 비교")
    print(f"   (썸네일 있는 상품: {total_comparisons}개 비교)")
    print(f"⚡ 병렬 처리: 최대 {max_workers}개 스레드 사용\n")
    
    completed = 0
    start_time = time.time()
    
    # 병렬 처리로 비교 수행
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # 각 비교 작업을 제출
        future_to_task = {
            executor.submit(calculate_combined_similarity, p1, p2): (p1, p2)
            for p1, p2 in comparison_tasks
        }
        
        # 완료된 작업 처리
        for future in as_completed(future_to_task):
            completed += 1
            p1, p2 = future_to_task[future]
            
            # 진행 상황 출력 (10%마다 또는 완료 시)
            if completed % max(1, total_comparisons // 10) == 0 or completed == total_comparisons:
                progress = (completed / total_comparisons) * 100
                elapsed = time.time() - start_time
                if completed > 0:
                    estimated_total = elapsed / (completed / total_comparisons)
                    remaining = max(0, estimated_total - elapsed)
                    print(f"  진행률: {progress:.1f}% ({completed}/{total_comparisons}) | "
                          f"경과: {elapsed/60:.1f}분 | 예상 남은 시간: {remaining/60:.1f}분")
            
            try:
                similarity = future.result()
                
                # 임계값 이상인 경우만 후보에 추가
                if similarity["combined_similarity"] >= threshold:
                    candidates.append({
                        "coupang_product": {
                            "title": p1.get("title", ""),
                            "price": p1.get("displayed_price") or p1.get("price") or p1.get("original_price", ""),
                            "thumbnail_url": p1.get("thumbnail_url", ""),
                            "product_link": p1.get("product_link", "")
                        },
                        "ssadagu_product": {
                            "title": p2.get("title", ""),
                            "price": p2.get("displayed_price") or p2.get("price") or p2.get("original_price", ""),
                            "thumbnail_url": p2.get("thumbnail_url", ""),
                            "product_link": p2.get("product_link", "")
                        },
                        "similarity": similarity
                    })
            except Exception as e:
                print(f"  ⚠️ 비교 실패: {str(e)[:50]}")
    
    elapsed_total = time.time() - start_time
    print(f"\n✅ 필터링 완료: {len(candidates)}개 후보 발견 (임계값: {threshold} 이상)")
    print(f"⏱️ 총 소요 시간: {elapsed_total/60:.1f}분")
    if elapsed_total > 0:
        print(f"📊 평균 처리 속도: {total_comparisons/(elapsed_total/60):.1f}개 비교/분\n")
    return candidates


print("✅ 종합 유사도 계산 및 필터링 함수 정의 완료")
print("   - calculate_combined_similarity(): 이미지 + 상품명 종합 유사도")
print("   - filter_similar_products(): 유사한 상품 쌍 필터링\n")


✅ 종합 유사도 계산 및 필터링 함수 정의 완료
   - calculate_combined_similarity(): 이미지 + 상품명 종합 유사도
   - filter_similar_products(): 유사한 상품 쌍 필터링



In [32]:
# Cell 6: JSON 파일 로드 및 비교 실행

# JSON 파일 경로 설정
coupang_json_path = "../crawling_tests/coupang_search_results.json"
ssadagu_json_path = "../crawling_tests/ssadagu_search_results.json"

# JSON 파일 로드
print("📂 JSON 파일 로드 중...\n")

with open(coupang_json_path, "r", encoding="utf-8") as f:
    coupang_data = json.load(f)

with open(ssadagu_json_path, "r", encoding="utf-8") as f:
    ssadagu_data = json.load(f)

coupang_products = coupang_data.get("products", [])
ssadagu_products = ssadagu_data.get("products", [])

print(f"✅ 쿠팡 상품: {len(coupang_products)}개")
print(f"✅ 싸다구 상품: {len(ssadagu_products)}개")
print(f"📊 검색 키워드: {coupang_data.get('search_keyword', 'N/A')}\n")

# 유사한 상품 필터링 실행
print("=" * 60)
similar_candidates = filter_similar_products(coupang_products, ssadagu_products)
print("=" * 60)


📂 JSON 파일 로드 중...

✅ 쿠팡 상품: 30개
✅ 싸다구 상품: 30개
📊 검색 키워드: 컴퓨터

🔍 총 30개(쿠팡) × 30개(싸다구) = 900개 비교
   (썸네일 있는 상품: 900개 비교)
⚡ 병렬 처리: 최대 10개 스레드 사용

  진행률: 10.0% (90/900) | 경과: 0.8분 | 예상 남은 시간: 6.9분
  진행률: 20.0% (180/900) | 경과: 1.0분 | 예상 남은 시간: 4.2분
  진행률: 30.0% (270/900) | 경과: 1.3분 | 예상 남은 시간: 3.1분
  진행률: 40.0% (360/900) | 경과: 1.6분 | 예상 남은 시간: 2.4분
  진행률: 50.0% (450/900) | 경과: 1.9분 | 예상 남은 시간: 1.9분
  진행률: 60.0% (540/900) | 경과: 2.3분 | 예상 남은 시간: 1.5분
  진행률: 70.0% (630/900) | 경과: 2.6분 | 예상 남은 시간: 1.1분
  진행률: 80.0% (720/900) | 경과: 2.9분 | 예상 남은 시간: 0.7분
  진행률: 90.0% (810/900) | 경과: 3.1분 | 예상 남은 시간: 0.3분
  진행률: 100.0% (900/900) | 경과: 3.4분 | 예상 남은 시간: 0.0분

✅ 필터링 완료: 10개 후보 발견 (임계값: 0.5 이상)
⏱️ 총 소요 시간: 3.4분
📊 평균 처리 속도: 264.6개 비교/분



In [33]:
# Cell 7: 결과 저장 및 출력

# 결과 저장
result = {
    "search_keyword": coupang_data.get("search_keyword", ""),
    "comparison_date": time.strftime("%Y-%m-%d %H:%M:%S"),
    "settings": {
        "similarity_threshold": SIMILARITY_THRESHOLD,
        "image_weight": IMAGE_WEIGHT,
        "price_weight": PRICE_WEIGHT,
        "title_weight": TITLE_WEIGHT
    },
    "statistics": {
        "total_coupang_products": len(coupang_products),
        "total_ssadagu_products": len(ssadagu_products),
        "total_comparisons": len(coupang_products) * len(ssadagu_products),
        "candidates_found": len(similar_candidates)
    },
    "candidates": similar_candidates
}

# JSON 파일로 저장
output_json = "compare_coupang_ssadagu_results.json"
with open(output_json, "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

print(f"✅ 결과 저장 완료: {output_json}\n")

# 결과 요약 출력
print("📊 비교 결과 요약:")
print(f"   - 쿠팡 상품 수: {len(coupang_products)}개")
print(f"   - 싸다구 상품 수: {len(ssadagu_products)}개")
print(f"   - 총 비교 횟수: {len(coupang_products) * len(ssadagu_products)}회")
print(f"   - 유사한 상품 쌍: {len(similar_candidates)}개 (임계값: {SIMILARITY_THRESHOLD} 이상)\n")

# 상위 10개 후보 출력
if similar_candidates:
    print("🔝 상위 유사 상품 쌍 (종합 유사도 순):")
    sorted_candidates = sorted(
        similar_candidates, 
        key=lambda x: x["similarity"]["combined_similarity"], 
        reverse=True
    )
    
    for idx, candidate in enumerate(sorted_candidates[:10], 1):
        sim = candidate["similarity"]
        print(f"\n  [{idx}] 종합 유사도: {sim['combined_similarity']:.2%}")
        print(
            f"      이미지: {sim['image_similarity']:.2%}, "
            f"가격: {sim['price_similarity']:.2%}, "
            f"상품명: {sim['title_similarity']:.2%}"
        )
        print(f"      쿠팡: {candidate['coupang_product']['title'][:50]}...")
        print(f"            가격: {candidate['coupang_product']['price']}")
        print(f"      싸다구: {candidate['ssadagu_product']['title'][:50]}...")
        print(f"              가격: {candidate['ssadagu_product']['price']}")
    
    if len(similar_candidates) > 10:
        print(f"\n  ... 외 {len(similar_candidates) - 10}개 더")
else:
    print("⚠️ 유사한 상품 쌍을 찾지 못했습니다.")
    print("   임계값을 낮추거나 Vision API로 추가 비교를 진행하세요.\n")


✅ 결과 저장 완료: compare_coupang_ssadagu_results.json

📊 비교 결과 요약:
   - 쿠팡 상품 수: 30개
   - 싸다구 상품 수: 30개
   - 총 비교 횟수: 900회
   - 유사한 상품 쌍: 10개 (임계값: 0.5 이상)

🔝 상위 유사 상품 쌍 (종합 유사도 순):

  [1] 종합 유사도: 58.96%
      이미지: 65.62%, 가격: 96.29%, 상품명: 33.63%
      쿠팡: 포유컴퓨터 조립 PC 게이밍 컴퓨터 풀세트 데스크탑 모니터 사무용 게임용 본체, PC02...
            가격: 798,000원
      싸다구: 2025 코어 i7-13620H 독립 그래픽 카드 4G 컴퓨터 비즈니스 사무용 디자인 게임...
              가격: 768,380원

  [2] 종합 유사도: 54.53%
      이미지: 56.25%, 가격: 98.60%, 상품명: 30.77%
      쿠팡: 윈도우11 설치 한컴오피스 정품 증정 사무용 가정용 컴퓨터 본체 데스크탑 PC...
            가격: 249,000원
      싸다구: T8PLUS 듀얼 기가비트 포트 3 HDMI2.0 N150 오피스 게임 4K 휴대용 소형 ...
              가격: 245,520원

  [3] 종합 유사도: 53.88%
      이미지: 71.88%, 가격: 79.69%, 상품명: 22.97%
      쿠팡: 삼성 컴퓨터 본체 데스크탑 윈도우11 가정 사무 포토샵 주식용 게임용, 6세대 G4400,...
            가격: 194,000원
      싸다구: 12G 독립 디스플레이 i3i5i7 인터넷 카페 e-스포츠 게임 데스크탑 컴퓨터 호스트 디...
              가격: 154,590원

  [4] 종합 유사도: 53.64%
      이미지: 68.75%, 가격: 92.73%, 상품명: 18.98%
      쿠팡: 삼성 사무용 컴퓨터 데스크탑 i5

In [ ]:
# Cell 8: Vision API로 상위 후보 정밀 비교

import os
import json
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI, RateLimitError, APIError

VISION_MODEL = "gpt-4o-mini"
VISION_DETAIL = "low"
TOP_K = 3
VISION_MAX_RETRIES = 3
VISION_BASE_WAIT = 2
VISION_RATE_LIMIT_WAIT = 10

# OpenAI 클라이언트 초기화
# .env 파일 경로 찾기 (더 안전한 방법)
current_file = Path.cwd()
project_root = current_file.parent.parent
env_path = project_root / ".env"

if not env_path.exists():
    print(f"⚠️ .env 파일을 찾을 수 없습니다: {env_path}")
    print("   프로젝트 루트에 .env 파일을 생성하고 OPENAI_API_KEY를 설정하세요.")
    OPENAI_API_KEY = None
else:
    env_loaded = load_dotenv(env_path)
    OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "").strip()

if not similar_candidates:
    print("⚠️ Vision 비교를 수행할 후보가 없습니다. 먼저 Cell 6을 실행하세요.")
elif not OPENAI_API_KEY:
    print("⚠️ OPENAI_API_KEY가 설정되어 있지 않습니다.")
    print(f"   프로젝트 루트의 .env 파일({env_path})에 다음과 같이 키를 추가하세요:")
    print("   OPENAI_API_KEY=sk-your-key-here")
elif not OPENAI_API_KEY.startswith("sk-"):
    print("⚠️ OPENAI_API_KEY 형식이 올바르지 않습니다. 'sk-'로 시작하는 키를 확인하세요.")
else:
    client = OpenAI(api_key=OPENAI_API_KEY)
    
    def parse_vision_response(raw_text: str):
        """```json 코드 블록을 포함한 응답을 안전하게 파싱합니다."""
        text = raw_text.strip()
        code_block = re.search(r"```(?:json)?\s*(.*?)```", text, re.DOTALL)
        if code_block:
            text = code_block.group(1).strip()
        try:
            parsed = json.loads(text)
            if isinstance(parsed, dict):
                parsed.setdefault("status", "ok")
            return parsed
        except json.JSONDecodeError:
            return None
    
    def call_vision_api_with_retry(content):
        last_error = None
        for attempt in range(VISION_MAX_RETRIES):
            try:
                response = client.chat.completions.create(
                    model=VISION_MODEL,
                    messages=content,
                    max_tokens=400,
                    timeout=40,
                )
                return response, None
            except RateLimitError as e:
                wait_time = VISION_RATE_LIMIT_WAIT * (attempt + 1)
                print(f"  ⚠️ Rate limit 발생. {wait_time}초 대기 후 재시도 ({attempt + 1}/{VISION_MAX_RETRIES})...")
                time.sleep(wait_time)
                last_error = f"Rate limit: {e}"
            except APIError as e:
                wait_time = VISION_BASE_WAIT ** (attempt + 1)
                print(f"  ⚠️ API 에러 발생. {wait_time}초 대기 후 재시도 ({attempt + 1}/{VISION_MAX_RETRIES})...")
                time.sleep(wait_time)
                last_error = f"API error: {e}"
            except Exception as e:
                if attempt == VISION_MAX_RETRIES - 1:
                    last_error = f"Unexpected error: {e}"
                    break
                wait_time = VISION_BASE_WAIT ** (attempt + 1)
                print(f"  ⚠️ 일시적 오류. {wait_time}초 대기 후 재시도 ({attempt + 1}/{VISION_MAX_RETRIES})...")
                time.sleep(wait_time)
        return None, last_error or "Vision API 호출 실패"
    
    def analyze_pair_with_vision(candidate, detail=VISION_DETAIL):
        """썸네일 2장을 Vision API로 비교하고 JSON 응답을 반환합니다."""
        coupang = candidate["coupang_product"]
        ssadagu = candidate["ssadagu_product"]
        coupang_thumb = coupang.get("thumbnail_url", "")
        ssadagu_thumb = ssadagu.get("thumbnail_url", "")
        if not coupang_thumb or not ssadagu_thumb:
            return {
                "status": "skipped",
                "reason": "missing_thumbnail",
                "message": "썸네일 URL이 없어 Vision 비교를 건너뜁니다."
            }, None
        prompt = f"""두 쇼핑몰 썸네일이 같은 제품을 나타내는지 비교하세요. 반드시 JSON으로만 답변하세요.
출력 형식:
{{
  \"isSameProduct\": \"y\" 또는 \"n\",
  \"confidence\": 0-100 사이 숫자,
  \"keySimilarities\": ["항목"],
  \"keyDifferences\": ["항목"],
  \"verdict\": "간단한 판단 이유"
}}
제품 정보:
- Coupang: {coupang.get('title', 'N/A')} / 가격 {coupang.get('price', 'N/A')}
- Ssadagu: {ssadagu.get('title', 'N/A')} / 가격 {ssadagu.get('price', 'N/A')}"""
        content = [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {
                        "type": "image_url",
                        "image_url": {"url": coupang_thumb, "detail": detail}
                    },
                    {
                        "type": "image_url",
                        "image_url": {"url": ssadagu_thumb, "detail": detail}
                    },
                ],
            }
        ]
        response, error_msg = call_vision_api_with_retry(content)
        if response is None:
            return {
                "status": "request_failed",
                "message": error_msg
            }, None
        raw_text = response.choices[0].message.content.strip()
        parsed = parse_vision_response(raw_text)
        if parsed is None:
            return {
                "status": "parse_error",
                "raw_response": raw_text
            }, raw_text
        return parsed, raw_text
    
    top_candidates = sorted(
        similar_candidates,
        key=lambda x: x["similarity"]["combined_similarity"],
        reverse=True,
    )[:TOP_K]
    
    vision_comparison_results = []
    print(f"🔎 Vision API 정밀 비교 시작 (상위 {len(top_candidates)}개 후보, detail='{VISION_DETAIL}')\n")
    for idx, candidate in enumerate(top_candidates, 1):
        coupang = candidate["coupang_product"]
        ssadagu = candidate["ssadagu_product"]
        combined = candidate["similarity"]["combined_similarity"]
        print("=" * 70)
        print(f"[{idx}] 종합 유사도 {combined:.2%}")
        print(f"  Coupang : {coupang['title'][:80]}...")
        print(f"             가격 {coupang['price']}")
        print(f"  Ssadagu : {ssadagu['title'][:80]}...")
        print(f"             가격 {ssadagu['price']}")
        
        result, raw_text = analyze_pair_with_vision(candidate)
        vision_comparison_results.append({
            "candidate": candidate,
            "vision_result": result,
            "raw_text": raw_text,
        })
        
        status = result.get("status")
        if status == "skipped":
            print(f"  ⚠️ Vision 비교 스킵: {result.get('message')}")
        elif status == "request_failed":
            print(f"  ⚠️ Vision API 호출 실패: {result.get('message')}")
        elif status == "parse_error":
            print("  ⚠️ JSON 파싱 실패. 원문 출력:")
            raw_preview = result.get('raw_response', '')
            print(f"     {raw_preview[:200]}...")
        else:
            print("  ✅ Vision 결과:")
            print(f"     동일 여부 : {result.get('isSameProduct', 'N/A')}, 신뢰도 {result.get('confidence', 'N/A')}%")
            similarities = result.get("keySimilarities", [])
            if similarities and isinstance(similarities, list):
                print(f"     공통점   : {', '.join(similarities[:3])}")
            differences = result.get("keyDifferences", [])
            if differences and isinstance(differences, list):
                print(f"     차이점   : {', '.join(differences[:3])}")
            if result.get("verdict"):
                print(f"     판단 사유: {result['verdict']}")
    
    export_payload = {
        "generated_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        "model": VISION_MODEL,
        "detail": VISION_DETAIL,
        "top_k": len(top_candidates),
        "results": []
    }
    for item in vision_comparison_results:
        export_payload["results"].append({
            "coupang_product": item["candidate"]["coupang_product"],
            "ssadagu_product": item["candidate"]["ssadagu_product"],
            "similarity": item["candidate"]["similarity"],
            "vision_result": item["vision_result"],
        })
    
    output_path = Path("vision_top3_results.json")
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(export_payload, f, ensure_ascii=False, indent=2)
    
    print("\n🎯 Vision 비교 완료. 결과는 vision_comparison_results 변수와 vision_top3_results.json 파일에 저장되었습니다.")
    print(f"📝 저장 경로: {output_path.resolve()}")


🔎 Vision API 정밀 비교 시작 (상위 3개 후보, detail='low')

[1] 종합 유사도 58.96%
  Coupang : 포유컴퓨터 조립 PC 게이밍 컴퓨터 풀세트 데스크탑 모니터 사무용 게임용 본체, PC02...
             가격 798,000원
  Ssadagu : 2025 코어 i7-13620H 독립 그래픽 카드 4G 컴퓨터 비즈니스 사무용 디자인 게임 16inch 게임 노트북...
             가격 768,380원
  ✅ Vision 결과:
     동일 여부 : n, 신뢰도 75%
     공통점   : 고성능 하드웨어, 게이밍 용도
     차이점   : PC와 노트북의 차이, 구성품의 차이 (모니터 포함 여부), 가격대 차이
     판단 사유: 제품 카테고리가 다르며, 하나는 조립형 PC이고 다른 하나는 노트북입니다.
[2] 종합 유사도 54.53%
  Coupang : 윈도우11 설치 한컴오피스 정품 증정 사무용 가정용 컴퓨터 본체 데스크탑 PC...
             가격 249,000원
  Ssadagu : T8PLUS 듀얼 기가비트 포트 3 HDMI2.0 N150 오피스 게임 4K 휴대용 소형 컴퓨터 VS N100...
             가격 245,520원
  ✅ Vision 결과:
     동일 여부 : n, 신뢰도 70%
     공통점   : 컴퓨터, 사무용, 윈도우11
     차이점   : 제품 모델, 가격, 기능
     판단 사유: 두 제품은 컴퓨터라는 점에서는 유사하지만 모델과 기능이 다른 점에서 동일하지 않음.
[3] 종합 유사도 53.88%
  Coupang : 삼성 컴퓨터 본체 데스크탑 윈도우11 가정 사무 포토샵 주식용 게임용, 6세대 G4400, WIN11 Pro, 240GB, 8GB, 블랙...
             가격 194,000원
  Ssadagu : 12G 독립 디스플레이 i3i5i7 인터넷 카페 e-스포츠 게임 데스크탑 컴퓨터 호스트 디자인 라이브 